We know from feature calculator and the linear regression the features which need to be considered (ladder_position, ladder_position_prev and percentage)

In [33]:
import pandas as pd

df = pd.read_csv("data/test.csv")

df["Year_Num"] = df["Year"].str.replace("x", "0").astype(int)
df["Percentage"] = df["Percentage"].fillna(100)
df["Ladder_Position"] = df["Ladder_Position"].fillna(df["Ladder_Position"].mean())
df["Ladder_Position_Prev"] = df["Ladder_Position_Prev"].fillna(df["Ladder_Position_Prev"].mean())

def predict_ladder_position_next_next(df):
    df = df.copy()

    required_cols = ["Ladder_Position", "Ladder_Position_Prev", "Percentage"]
    if not all(col in df.columns for col in required_cols):
        raise ValueError("Missing one or more required columns.")

    df["Change"] = df["Ladder_Position"] - df["Ladder_Position_Prev"]

    def adjust_change(row):
        percentage = row["Percentage"] / 100
        change = row["Change"]
        factor = percentage if percentage > 1 else (2 - percentage)
        return change * factor

    df["Adjusted_Change"] = df.apply(adjust_change, axis=1)

    df["Ladder_Position_Next"] = df["Ladder_Position"] + df["Adjusted_Change"]

    return df[[
        "ID", "Year", "Ladder_Position", "Ladder_Position_Prev",
        "Percentage", "Ladder_Position_Next"
    ]]

predicted_df = predict_ladder_position_next_next(df)
predicted_df['Ladder_Position_Next'] = predicted_df["Ladder_Position_Next"].clip(lower = 1)

print(predicted_df[['Ladder_Position_Next']])

predicted_df[["ID", "Ladder_Position_Next"]].to_csv("submission.csv", index=False)

     Ladder_Position_Next
0                  5.9232
1                  6.1093
2                  1.0000
3                  1.0000
4                  1.0000
..                    ...
469               24.7610
470                3.9301
471                1.0000
472               18.9254
473                1.9760

[474 rows x 1 columns]
